____
# Case Study 1: Aqueous Solvation Free Energy of Neutral Molecules
____

This tutorial demonstrates how to use the `SolvationFepDriver` in VeloxChem to compute the absolute aqueous solvation free energy of a neutral organic molecule. 

In this example, we will calculate the hydration free energy of **aspirin**. The driver will autonomously handle:
1. Generating the solvent box (SPC/E water).
2. Parameterizing the solute (GAFF force field with RESP partial charges).
3. Orchestrating the 4-stage alchemical decoupling using a Gaussian soft-core (GSC) potential.
4. Running the molecular dynamics using OpenMM.
5. Evaluating the final free energy using PyMBAR.

## 1. Define the Solute
We initialize our solute molecule using VeloxChem's built-in molecule reader. You can provide a common chemical name, a SMILES string, or an optimized XYZ coordinate file.

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name('aspirin')

## 2. Initialize the SolvationFepDriver
Next, we instantiate the `SolvationFepDriver`. By default, the driver is configured for aqueous solvation at standard conditions (298.15 K, 1 atm). 

*Note: While we use the default settings here, you can easily access the driver's properties to modify simulation lengths, the number of lambda windows, or the target solvent.*

In [ ]:
solvationFEP = vlx.SolvationFepDriver()

## 3. Execute the FEP Workflow
We execute the automated pipeline by passing our molecule to the `compute()` method. 

Behind the scenes, this will launch the OpenMM engine on the available GPU, run the equilibration and production dynamics across all intermediate states, and process the thermodynamic data. 

*(Depending on your hardware, this single line of code will take approximately 10 minutes to complete).*

In [ ]:
# Settings are here chosen for a quick execution,
# it is recommended to use the default settings
solvationFEP.num_steps = 5000
solvationFEP.num_snapshots = 50
results = solvationFEP.compute(molecule)

## 4. Analyze the Results
The `compute()` method returns a dictionary containing the computed free energy differences ($\Delta G$) and their associated statistical uncertainties for each stage of the thermodynamic cycle. 

Let's extract and print the final absolute solvation free energy.

In [ ]:
import math

# Extract final free energy (value is returned in kJ/mol)
dG_solv = results['free_energy']

# Propagate error from the 4 FEP stages (sqrt of sum of variances)
variance_sum = sum(results[f"Stage {i}"]['Uncertainty']**2 for i in range(1, 5))
error = math.sqrt(variance_sum)

print(f"Absolute Solvation Free Energy of Aspirin: {dG_solv:.2f} ± {error:.2f} kJ/mol")